In [1]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict
import time

In [2]:
class CrashState(TypedDict):
    input: str
    step1: str
    step2: str
    step3: str

In [3]:
def step1(state:CrashState) -> CrashState:
    print("Step1 Executed")
    return {"step1":"Done", "input":state["input"]}

def step2(state: CrashState) -> CrashState:
    print("Step2 Hanging... now manually interrupt from the notebook toolbar (STOP button)")
    time.sleep(30)
    return {"step2":"Done"}

def step3(state: CrashState) ->CrashState:
    print("Step3 Executed")
    return {"done":True}


In [4]:
#Build the Graph
builder = StateGraph(CrashState)

builder.add_node("step1", step1)
builder.add_node("step2", step2)
builder.add_node("step3", step3)

builder.add_edge(START, "step1")
builder.add_edge("step1", "step2")
builder.add_edge("step2", "step3")
builder.add_edge("step3", END)

checkpointer = InMemorySaver()

graph = builder.compile(checkpointer=checkpointer)

In [5]:
try:
    print("Running Graph: Please manually interrupt during Step2...")
    graph.invoke({"input":"start"}, config={"configurable":{"thread_id":'thread-1'}})
except KeyboardInterrupt:
    print("Kernel Manually interrupted (Crash Simulated).")


Running Graph: Please manually interrupt during Step2...
Step1 Executed
Step2 Hanging... now manually interrupt from the notebook toolbar (STOP button)
Kernel Manually interrupted (Crash Simulated).


In [6]:
graph.get_state({"configurable":{"thread_id":'thread-1'}})

StateSnapshot(values={'input': 'start', 'step1': 'Done'}, next=('step2',), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f097c32-e4d6-619e-8001-626fbe7696be'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}, 'thread_id': 'thread-1'}, created_at='2025-09-22T14:48:19.972547+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f097c32-e4d0-6513-8000-403e480bb13c'}}, tasks=(PregelTask(id='cff84028-2d56-62cc-ad43-3f7a4dba0de2', name='step2', path=('__pregel_pull', 'step2'), error=None, interrupts=(), state=None, result=None),), interrupts=())

In [7]:
final_state = graph.invoke(None, config={"configurable":{"thread_id":'thread-1'}})
print("\n Final State:", final_state)

Step2 Hanging... now manually interrupt from the notebook toolbar (STOP button)
Step3 Executed

 Final State: {'input': 'start', 'step1': 'Done', 'step2': 'Done'}
